# Task 4: Two RGB seeds and green-feature fusion

Attach only the official 2026task-4 competition data and enable GPU. No pretrained weights, previous Notebook Outputs or external datasets are required. Internet is needed only if a dependency must be installed.

Training and submission files are written under `/kaggle/working/task4/`. Keep the complete Notebook Output to retain fold checkpoints and probability files.

In [ ]:
from pathlib import Path
import torch
if not Path("/kaggle/working").is_dir() or not torch.cuda.is_available():
    raise RuntimeError("Run this notebook on Kaggle with GPU enabled.")


In [ ]:
%pip install -q "numpy>=1.26" "pandas>=2.2" "Pillow>=10" "scipy>=1.12" "scikit-learn>=1.4" "opencv-python-headless>=4.8"


In [ ]:
Path("/kaggle/working/task4_source").mkdir(exist_ok=True)


In [ ]:
%%writefile /kaggle/working/task4_source/runtime.py
"""Official-data loading, patch U-Nets, baseline training and exact competition output protocol."""
from __future__ import annotations
import ast
import contextlib
import copy
import hashlib
import json
import math
import os
import platform
import random
import time
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from sklearn.model_selection import KFold
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

VERSION = 'vessel-segmentation'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cv2.setNumThreads(0)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf8')
    tmp.replace(path)


def digest_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def markdown_table(frame, index=False):
    frame = frame.reset_index() if index else frame
    rows = [[str(x).replace('|', '\\|') for x in frame.columns]]
    rows += [['---'] * len(frame.columns)]
    for row in frame.itertuples(index=False, name=None):
        rows.append([f'{x:.6g}' if isinstance(x, float) else str(x).replace('|', '\\|') for x in row])
    return '\n'.join('| ' + ' | '.join(row) + ' |' for row in rows)


def find_data(root=None):
    if root is not None:
        root = Path(root)
        candidates = [root] if (root / 'sample_submission.csv').exists() else list(root.rglob('sample_submission.csv'))
        candidates = [x if x.is_dir() else x.parent for x in candidates]
    else:
        candidates = [p.parent for p in Path('/kaggle/input').rglob('sample_submission.csv')
                      if (p.parent / 'segmentation_to_csv.py').exists()
                      and (p.parent / 'train/image').is_dir()]
    assert len(candidates) == 1, f'须仅挂载本比赛官方数据；找到 {candidates}。可设置 DATA_ROOT。'
    root = candidates[0]
    assert all((root / p).exists() for p in ['train/image', 'train/label', 'test/image', 'segmentation_to_csv.py'])
    return root


class OfficialData:
    def __init__(self, root, out):
        self.root, self.out = Path(root), Path(out)
        self.out.mkdir(parents=True, exist_ok=True)
        self.sample = pd.read_csv(self.root / 'sample_submission.csv', dtype={'Id': str}, keep_default_na=False)
        assert self.sample.columns.tolist() == ['Id', 'Predicted']
        self.ids = sorted([p.stem for p in (self.root / 'train/image').glob('*.jpg')], key=int)
        self.test_ids = self.sample.Id.tolist()
        assert len(self.ids) == len(self.test_ids) == 20
        assert len(set(self.ids)) == len(set(self.test_ids)) == 20
        self.images, self.labels, self.proxy, self.test_images, self.rows = [], [], [], [], []
        pixel_hashes = []
        for split, ids in [('train', self.ids), ('test', self.test_ids)]:
            for name in ids:
                p = self.root / split / 'image' / f'{name}.jpg'
                a = np.asarray(Image.open(p).convert('RGB')).copy()
                assert a.shape == (584, 565, 3), (p, a.shape)
                pixel_hash = hashlib.sha256(a.tobytes()).hexdigest()
                pixel_hashes.append(pixel_hash)
                row = dict(split=split, id=name, height=a.shape[0], width=a.shape[1],
                           sha256=digest_file(p), pixel_sha256=pixel_hash)
                if split == 'train':
                    lp = self.root / 'train/label' / f'{name}.jpg'
                    im = Image.open(lp).convert('L')
                    label = np.asarray(im)
                    assert label.shape == a.shape[:2] and set(np.unique(label)).issubset({0, 255})
                    y = (label > 127).astype(np.uint8)
                    assert .01 < y.mean() < .3, '标签约定变化：停止并重新核对前景。'
                    proxy = (np.asarray(im.resize((512, 512), Image.Resampling.BILINEAR)) > 0).astype(np.uint8)
                    self.images.append(a)
                    self.labels.append(y)
                    self.proxy.append(proxy)
                    row.update(label_sha256=digest_file(lp), vessel_fraction=float(y.mean()),
                               proxy_fraction=float(proxy.mean()))
                else:
                    self.test_images.append(a)
                self.rows.append(row)
        assert len(pixel_hashes) == len(set(pixel_hashes)), '发现重复原图，需要按重复组重新划分。'
        self.folds = np.full(20, -1, dtype=int)
        for fold, (_, va) in enumerate(KFold(5, shuffle=True, random_state=3407).split(self.ids)):
            self.folds[va] = fold
        assert np.all(np.bincount(self.folds) == 4)
        pd.DataFrame(self.rows).to_csv(self.out / 'data_audit.csv', index=False)
        pd.DataFrame({'Id': self.ids, 'fold': self.folds}).to_csv(self.out / 'folds.csv', index=False)
        # Dataset hash includes labels, sample order and the official converter.
        self.fingerprint = hashlib.sha256(json.dumps(self.rows, sort_keys=True).encode()
            + (self.root / 'sample_submission.csv').read_bytes()
            + (self.root / 'segmentation_to_csv.py').read_bytes()).hexdigest()
        self.cache = {}
        print('官方数据：20 train / 20 test；584×565；白色血管；按原图固定五折。')

    def prepared(self, mode, test=False):
        key = (mode, test)
        if key not in self.cache:
            self.cache[key] = [preprocess(x, mode) for x in (self.test_images if test else self.images)]
        return self.cache[key]



def preprocess(rgb, mode):
    if mode == 'rgb':
        return rgb.copy()
    g = rgb[..., 1]
    if mode == 'green_raw':
        return np.repeat(g[..., None], 3, axis=2)
    assert mode == 'green'
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
    contrast = np.clip(4 * g.astype(np.float32) - 4 * cv2.GaussianBlur(g, (0, 0), 8).astype(np.float32) + 128, 0, 255).astype(np.uint8)
    return np.stack([g, clahe, contrast], axis=-1)


@dataclass(frozen=True)
class Recipe:
    name: str = 'resunet_green'
    arch: str = 'resunet'
    mode: str = 'green'
    width: int = 24
    patch: int = 192
    lr: float = 8e-4
    wd: float = 1e-4
    dice_weight: float = .5
    augmentation: bool = True
    multiscale: bool = True
    ema: bool = True
    context: bool = False
    seed: int = 20261031


class PatchDataset(Dataset):
    def __init__(self, images, labels, indices, recipe, count, seed):
        self.x, self.y, self.indices, self.cfg = images, labels, list(indices), recipe
        self.count, self.seed = count, seed
        # Purely uniform crops avoid changing the class prior through label-conditioned sampling.

    def __len__(self): return self.count

    def __getitem__(self, index):
        rng = np.random.default_rng(self.seed + index)
        i = self.indices[int(rng.integers(len(self.indices)))]
        x, y = self.x[i], self.y[i]
        p = self.cfg.patch
        scale = float(rng.choice([.8, 1., 1.25])) if self.cfg.multiscale else 1.
        span = p * scale
        cx = float(rng.uniform(span / 2, x.shape[1] - span / 2))
        cy = float(rng.uniform(span / 2, x.shape[0] - span / 2))
        angle = float(rng.uniform(-180, 180)) if self.cfg.augmentation else 0.
        # WARP_INVERSE_MAP maps each output patch pixel into the source image.
        a = math.radians(angle)
        mat = np.array([[scale * math.cos(a), -scale * math.sin(a), 0],
                        [scale * math.sin(a), scale * math.cos(a), 0]], dtype=np.float32)
        mat[:, 2] = [cx, cy] - mat[:, :2] @ np.array([(p - 1) / 2, (p - 1) / 2])
        x = cv2.warpAffine(x, mat, (p, p), flags=cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP,
                           borderMode=cv2.BORDER_REFLECT_101)
        y = cv2.warpAffine(y, mat, (p, p), flags=cv2.INTER_NEAREST | cv2.WARP_INVERSE_MAP,
                           borderMode=cv2.BORDER_REFLECT_101)
        if self.cfg.augmentation:
            if rng.random() < .5: x, y = x[:, ::-1], y[:, ::-1]
            if rng.random() < .5: x, y = x[::-1], y[::-1]
        x = x.astype(np.float32) / 255.
        if self.cfg.augmentation:
            x = np.clip(x * rng.uniform(.85, 1.15) + rng.uniform(-.06, .06), 0, 1)
            if rng.random() < .25:
                x = np.clip(x + rng.normal(0, .012, x.shape).astype(np.float32), 0, 1)
        return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1) * 2 - 1)), torch.from_numpy(np.ascontiguousarray(y[None])).float()


class Block(nn.Module):
    def __init__(self, cin, cout, residual):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.SiLU(inplace=True),
                                  nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout))
        self.skip = (nn.Identity() if cin == cout else nn.Conv2d(cin, cout, 1, bias=False)) if residual else None

    def forward(self, x):
        y = self.body(x)
        return F.silu(y if self.skip is None else y + self.skip(x), inplace=True)


class VesselUNet(nn.Module):
    def __init__(self, recipe):
        super().__init__()
        assert recipe.arch in {'unet', 'resunet'}
        w, residual = recipe.width, recipe.arch == 'resunet'
        self.enc = nn.ModuleList([Block(3, w, residual), Block(w, 2*w, residual), Block(2*w, 4*w, residual),
                                  Block(4*w, 8*w, residual), Block(8*w, 16*w, residual)])
        self.context = nn.ModuleList([nn.Conv2d(16*w, 16*w, 3, padding=d, dilation=d, groups=16*w)
                                      for d in [1, 2, 4]]) if recipe.context else None
        self.up = nn.ModuleList([nn.Conv2d(16*w, 8*w, 1), nn.Conv2d(8*w, 4*w, 1), nn.Conv2d(4*w, 2*w, 1), nn.Conv2d(2*w, w, 1)])
        self.dec = nn.ModuleList([Block(16*w, 8*w, residual), Block(8*w, 4*w, residual),
                                  Block(4*w, 2*w, residual), Block(2*w, w, residual)])
        self.head = nn.Conv2d(w, 1, 1)

    def forward(self, x):
        skips = []
        for j, block in enumerate(self.enc):
            if j: x = F.max_pool2d(x, 2)
            x = block(x); skips.append(x)
        if self.context is not None:
            x = x + sum(layer(x) for layer in self.context) / len(self.context)
        for j, (up, block) in enumerate(zip(self.up, self.dec)):
            s = skips[-2-j]
            x = up(F.interpolate(x, size=s.shape[-2:], mode='bilinear', align_corners=False))
            x = block(torch.cat([x, s], dim=1))
        return self.head(x)


def loss_fn(logits, target, dice_weight):
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = logits.sigmoid()
    dice = 1 - ((2 * (p * target).sum((1, 2, 3)) + 1) /
                (p.sum((1, 2, 3)) + target.sum((1, 2, 3)) + 1)).mean()
    return (1 - dice_weight) * bce + dice_weight * dice


def autocast():
    return torch.autocast('cuda', dtype=torch.float16) if DEVICE.type == 'cuda' else contextlib.nullcontext()


@torch.no_grad()
def predict(model, image, tta=False):
    model.eval()
    x = torch.from_numpy(image.transpose(2, 0, 1).copy()).float()[None].to(DEVICE) / 127.5 - 1
    h, w = x.shape[-2:]
    x = F.pad(x, (0, (-w) % 16, 0, (-h) % 16), mode='reflect')
    variants = [(), (-1,), (-2,), (-2, -1)] if tta else [()]
    probs = []
    for dims in variants:
        z = x.flip(dims) if dims else x
        with autocast(): p = model(z).float().sigmoid()
        if dims: p = p.flip(dims)
        probs.append(p[0, 0, :h, :w].cpu().numpy())
    return np.mean(probs, axis=0).astype(np.float32)


def dice_score(y, m):
    y, m = np.asarray(y, bool), np.asarray(m, bool)
    return float((2 * np.logical_and(y, m).sum() + 1e-8) / (y.sum() + m.sum() + 1e-8))


def proxy_prob(p):
    # Resize probabilities once, then binarize on the exact submission grid.
    return cv2.resize(p.astype(np.float32), (512, 512), interpolation=cv2.INTER_LINEAR)


def validate(model, data, indices, mode):
    native, proxy = [], []
    images = data.prepared(mode)
    for i in indices:
        p = predict(model, images[i])
        native.append(dice_score(data.labels[i], p >= .5))
        proxy.append(dice_score(data.proxy[i], output_mask(p, .5, 0, 'native_then_official')))
    return float(np.mean(native)), float(np.mean(proxy))


def signature(data, recipe, fold, steps, batch, code_hash, eval_every):
    spec = dict(version=VERSION, data=data.fingerprint, recipe=asdict(recipe), fold=fold,
                steps=steps, batch=batch, code_hash=code_hash, eval_every=eval_every,
                torch=torch.__version__, device=DEVICE.type)
    return hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest(), spec


def train_fold(data, recipe, fold, steps, batch, out, code_hash, workers=2, eval_every=250):
    """Checkpoints are reusable only for an exact matching, completed run."""
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    sig, spec = signature(data, recipe, fold, steps, batch, code_hash, eval_every)
    path = out / 'best.pt'
    done = out / 'completed.json'
    if done.exists():
        meta = json.loads(done.read_text())
        assert meta['signature'] == sig and path.exists(), f'输出目录含不同实验：{out}，请换目录。'
        return path, meta
    if (out / 'spec.json').exists():
        assert json.loads((out / 'spec.json').read_text())['signature'] == sig, f'禁止混用旧实验目录：{out}'
    write_json(out / 'spec.json', dict(signature=sig, spec=spec))
    tr = np.flatnonzero(data.folds != fold); va = np.flatnonzero(data.folds == fold)
    assert not set(tr) & set(va) and len(tr) == 16 and len(va) == 4
    seed_all(recipe.seed)
    model = VesselUNet(recipe).to(DEVICE)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    ds = PatchDataset(data.prepared(recipe.mode), data.labels, tr, recipe, steps * batch, recipe.seed + fold * 1000003)
    loader = DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers,
                        pin_memory=DEVICE.type == 'cuda', persistent_workers=workers > 0)
    optim = torch.optim.AdamW(model.parameters(), lr=recipe.lr, weight_decay=recipe.wd)
    scaler = torch.amp.GradScaler('cuda', enabled=DEVICE.type == 'cuda')
    warmup = min(100, max(1, steps // 10))
    history, best, best_step, start = [], -1., 0, time.time()
    ema_state = ema.state_dict()
    nparams = sum(p.numel() for p in model.parameters())
    print(f'{recipe.name} fold={fold} steps={steps} params={nparams:,}', flush=True)
    for step, (x, y) in enumerate(loader, 1):
        model.train()
        frac = max(0., (step - warmup) / max(1, steps - warmup))
        lr = recipe.lr * (step / warmup if step <= warmup else .05 + .95 * .5 * (1 + math.cos(math.pi * frac)))
        for group in optim.param_groups: group['lr'] = lr
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optim.zero_grad(set_to_none=True)
        with autocast(): loss = loss_fn(model(x), y, recipe.dice_weight)
        assert torch.isfinite(loss), '训练 loss 非有限值。'
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        nn.utils.clip_grad_norm_(model.parameters(), 5.)
        scaler.step(optim); scaler.update()
        # Copy BN running statistics; averaging them would add a second lag.
        decay = min(.995, (step - 1) / (step + 9))
        with torch.no_grad():
            for key, value in model.state_dict().items():
                if value.is_floating_point() and 'running_' not in key:
                    ema_state[key].mul_(decay).add_(value, alpha=1-decay)
                else: ema_state[key].copy_(value)
        if step % eval_every == 0 or step == steps:
            chosen = ema if recipe.ema else model
            native, proxy = validate(chosen, data, va, recipe.mode)
            row = dict(step=step, loss=float(loss.detach().cpu()), lr=lr, native_dice=native,
                       proxy_dice=proxy, elapsed_seconds=time.time()-start)
            history.append(row)
            pd.DataFrame(history).to_csv(out / 'history.csv', index=False)
            print(f'  {step:5d} loss={row["loss"]:.4f} native={native:.5f} proxy={proxy:.5f} seconds={row["elapsed_seconds"]:.0f}', flush=True)
            if proxy > best:
                best, best_step = proxy, step
                payload = dict(signature=sig, recipe=asdict(recipe), fold=fold, step=step,
                               model={k: v.detach().cpu() for k, v in chosen.state_dict().items()},
                               raw={k: v.detach().cpu() for k, v in model.state_dict().items()},
                               spec=spec)
                tmp = path.with_suffix('.tmp'); torch.save(payload, tmp); tmp.replace(path)
    meta = dict(signature=sig, best_proxy_dice=best, best_step=best_step, seconds=time.time()-start,
                parameters=nparams, recipe=asdict(recipe), fold=int(fold))
    write_json(done, meta)
    del loader, model, ema, optim
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    return path, meta


def load_model(path, raw=False):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    model = VesselUNet(Recipe(**ck['recipe'])).to(DEVICE)
    model.load_state_dict(ck['raw' if raw else 'model'])
    return model.eval(), ck


def rle_encode(mask):
    """Exactly the official C-order, one-based encoding of nonzero pixels."""
    a = np.asarray(mask, dtype=np.uint8).ravel(order='C')
    padded = np.r_[0, a, 0]
    edges = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    edges[1::2] -= edges[::2]
    return ' '.join(map(str, edges))


def rle_decode(s, shape=(512, 512)):
    a = np.zeros(int(np.prod(shape)), np.uint8)
    words = np.array([int(v) for v in str(s).split()], dtype=np.int64)
    assert len(words) % 2 == 0
    prev = 0
    for start, length in words.reshape(-1, 2):
        assert length > 0 and start >= 1 and start - 1 >= prev and start - 1 + length <= a.size
        a[start-1:start-1+length] = 1; prev = start-1+length
    return a.reshape(shape)


def verify_official_encoder(root, out):
    """Execute official function definitions, without its main filesystem operations."""
    import torchvision
    source = (Path(root) / 'segmentation_to_csv.py').read_text(encoding='utf-8-sig')
    tree = ast.parse(source)
    defs = ast.Module(body=[x for x in tree.body if isinstance(x, ast.FunctionDef)], type_ignores=[])
    scope = dict(np=np, pd=pd, Image=Image, torchvision=torchvision, os=os)
    exec(compile(defs, 'official_segmentation_to_csv.py', 'exec'), scope)
    assert 'turn_to_str' in scope, '官方转换脚本发生变化，请重新核对。'
    out = Path(out) / 'encoder_check'; out.mkdir(exist_ok=True, parents=True)
    rng = np.random.default_rng(7)
    checks = [np.zeros((512, 512), np.uint8), np.ones((512, 512), np.uint8),
              (rng.random((512, 512)) < .03).astype(np.uint8)]
    p = np.zeros((512, 512), np.uint8); p[0, 0] = p[-1, -1] = p[13, 77] = 1; checks.append(p)
    paths = []
    for i, a in enumerate(checks):
        path = out / f'{i}.png'; Image.fromarray(a * 255).save(path); paths.append(str(path))
    reference = scope['turn_to_str'](paths)
    for a, expected in zip(checks, reference):
        assert rle_encode(a) == expected
        assert np.array_equal(rle_decode(expected), a)
    write_json(out / 'verified.json', dict(converter_sha256=digest_file(Path(root) / 'segmentation_to_csv.py'),
                                          cases=len(checks), order='C', first_index=1, positive='white/nonzero', shape=[512,512]))
    print('已与官方 turn_to_str 逐字比对：空图、满图、随机图、首尾像素全部一致。')


def component_filter(mask, min_size):
    if min_size == 0: return np.asarray(mask, bool)
    labels, n = ndimage.label(mask, structure=np.ones((3, 3)))
    counts = np.bincount(labels.ravel()); keep = counts >= min_size; keep[0] = False
    return keep[labels]


def output_mask(prob, threshold, min_size=0, output_protocol='probability512', shape=(512,512)):
    h,w = shape
    if output_protocol == 'native_then_official':
        native = Image.fromarray((np.asarray(prob) >= threshold).astype(np.uint8)*255)
        mask = np.asarray(native.resize((w,h),Image.Resampling.BILINEAR)) > 0
    else:
        assert output_protocol == 'probability512'
        mask = cv2.resize(np.asarray(prob,np.float32),(w,h),interpolation=cv2.INTER_LINEAR) >= threshold
    return component_filter(mask, min_size)


def score_at(probs, targets, threshold, min_size=0, output_protocol='probability512'):
    return np.array([dice_score(y, output_mask(p,threshold,min_size,output_protocol,y.shape)) for p, y in zip(probs, targets)])


def screening_recipes():
    base = Recipe()
    return [
        replace(base, name='baseline_unet_rgb_bce', arch='unet', mode='rgb', dice_weight=0., augmentation=False, multiscale=False, ema=False),
        base,
        replace(base, name='model_plain_unet', arch='unet'),
        replace(base, name='model_context', context=True),
        replace(base, name='input_rgb', mode='rgb'),
        replace(base, name='input_green_raw', mode='green_raw'),
        replace(base, name='ablation_no_augmentation', augmentation=False),
        replace(base, name='ablation_no_multiscale', multiscale=False),
        replace(base, name='ablation_bce_only', dice_weight=0.),
        replace(base, name='lr_4e4', lr=4e-4),
        replace(base, name='lr_16e4', lr=1.6e-3),
        replace(base, name='dice_weight_025', dice_weight=.25),
        replace(base, name='dice_weight_075', dice_weight=.75),
        replace(base, name='patch_128', patch=128),
        replace(base, name='patch_256', patch=256),
        replace(base, name='width_32', width=32),
    ]


def save_paired_analysis(data, out, results):
    df = pd.DataFrame(results).sort_values('proxy_dice', ascending=False)
    df.to_csv(out / 'screening.csv', index=False)
    base = df[df.name == 'resunet_green'].iloc[0]
    pairs = []
    for name, factor in [('model_plain_unet', 'residual blocks'), ('input_green_raw', 'CLAHE + local contrast'),
                         ('input_rgb', 'green feature representation'), ('ablation_no_augmentation', 'geometric/photometric augmentation'),
                         ('ablation_no_multiscale', 'multiscale crops'), ('ablation_bce_only', 'Dice loss')]:
        row = df[df.name == name].iloc[0]
        pairs.append(dict(factor=factor, full_recipe=float(base.proxy_dice), removed=float(row.proxy_dice),
                          difference=float(base.proxy_dice-row.proxy_dice), interpretation='single split screening; not causal proof'))
    pd.DataFrame(pairs).to_csv(out / 'ablation.csv', index=False)
    return df


def run_analysis(data, out, code_hash, steps=1000, confirm_steps=2000, batch=12, workers=2):
    out = Path(out); out.mkdir(exist_ok=True, parents=True)
    recipes = screening_recipes()
    results = []
    for cfg in recipes:
        path, meta = train_fold(data, cfg, 0, steps, batch, out / cfg.name / 'fold0', code_hash, workers)
        model, ck = load_model(path)
        va = np.flatnonzero(data.folds == 0)
        native, proxy = validate(model, data, va, cfg.mode)
        results.append(dict(**asdict(cfg), native_dice=native, proxy_dice=proxy,
                            seconds=meta['seconds'], parameters=meta['parameters'], best_step=meta['best_step']))
        del model
        pd.DataFrame(results).to_csv(out / 'screening_partial.csv', index=False)
    df = save_paired_analysis(data, out, results)
    # EMA and TTA are evaluated on the SAME training trajectory/checkpoint.
    path = out / 'resunet_green/fold0/best.pt'
    paired = []
    for raw in [True, False]:
        model, _ = load_model(path, raw=raw)
        for tta in [False, True]:
            for i in np.flatnonzero(data.folds == 0):
                p = predict(model, data.prepared('green')[i], tta)
                paired.append(dict(Id=data.ids[i], ema=not raw, tta=tta,
                    native_dice=dice_score(data.labels[i], p >= .5),
                    proxy_dice=dice_score(data.proxy[i], output_mask(p,.5,0,'native_then_official'))))
        del model
    pd.DataFrame(paired).to_csv(out / 'paired_ema_tta.csv', index=False)
    # New images for confirmation, and longer equal budgets for shortlisted models.
    shortlist = [next(c for c in recipes if c.name == name) for name in df.head(3).name]
    confirmed = []
    for cfg in shortlist:
        for fold in [1, 2]:
            path, meta = train_fold(data, cfg, fold, confirm_steps, batch,
                                    out / 'confirmation' / cfg.name / f'fold{fold}', code_hash, workers)
            model, _ = load_model(path)
            scores = []
            for i in np.flatnonzero(data.folds == fold):
                p = predict(model, data.prepared(cfg.mode)[i], True)
                scores.append(dice_score(data.proxy[i], output_mask(p,.5,0,'native_then_official')))
            confirmed.append(dict(name=cfg.name, fold=fold, dice=float(np.mean(scores)), seconds=meta['seconds']))
            del model
    conf = pd.DataFrame(confirmed); conf.to_csv(out / 'confirmation.csv', index=False)
    ranking = conf.groupby('name').dice.agg(['mean','std']).sort_values('mean', ascending=False)
    ranking.to_csv(out / 'confirmation_ranking.csv')
    winner = next(c for c in recipes if c.name == ranking.index[0])
    # Diversity candidate is confirmed rather than automatically assumed beneficial.
    ranked_recipes = [next(c for c in recipes if c.name == name) for name in ranking.index]
    second = next((c for c in ranked_recipes if c.mode != winner.mode), None)
    if second is None:
        second = replace(winner, name=winner.name+'_seed2', seed=20261101)
    selected = [asdict(winner), asdict(second)]
    write_json(out / 'selected_recipes.json', selected)
    print('筛选推荐配置已保存至 selected_recipes.json；正式入口使用固定的最终方案。')
    print(json.dumps(selected, ensure_ascii=False, indent=2))
    # All these are model-selection results, not an unbiased final score estimate.
    text = f'''# 课设实验记录（由实际运行生成）\n\n官方数据20张训练图、20张测试图。所有模型从零训练；按原图固定五折。\n\n共比较{len(recipes)}种设定，筛选预算每项{steps}步、batch={batch}。\n短名单在另外两折以每项{confirm_steps}步复核。筛选和复核分数参与了选模，不能作为无偏测试成绩。\n\n官方标签白色表示血管。原图标签为584×565，官方脚本先双线性缩放至512×512再取非零像素，正例边缘会扩展。\n本实验同时记录原始标签Dice和转换协议代理Dice；隐藏评分标签未公开，代理并不等于已验证的Kaggle分数。\n\n## 模型比较\n\n{markdown_table(df[['name','native_dice','proxy_dice','parameters','seconds']])}\n\n## 独立折复核\n\n{markdown_table(ranking,index=True)}\n\n消融结果见ablation.csv；同轨迹EMA/TTA见paired_ema_tta.csv；学习曲线位于各实验history.csv。\n差值来自有限的训练图；正差值只支持当前划分和预算下的有效性。\n'''
    (out / '实验报告.md').write_text(text, encoding='utf8')
    return selected


def make_submission(data, probs, calibration, out):
    probs = np.asarray(probs)
    if (probs.shape != (20, 584, 565) or not np.isfinite(probs).all()
            or ((probs < 0) | (probs > 1)).any()):
        raise ValueError('Expected twenty finite native-resolution probability maps in [0, 1]')
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    masks = [output_mask(p,calibration['threshold'],calibration['min_size'],
                          calibration.get('output_protocol','probability512')) for p in probs]
    df = data.sample.copy()
    df['Predicted'] = [rle_encode(m) for m in masks]
    assert df.Id.tolist() == data.test_ids and len(df) == 20 and not df.Predicted.isna().any()
    pred_dir = out / 'prediction_png'; pred_dir.mkdir(exist_ok=True)
    for name, mask, encoded in zip(df.Id, masks, df.Predicted):
        assert np.array_equal(rle_decode(encoded), mask)
        Image.fromarray(mask.astype(np.uint8) * 255).save(pred_dir / f'{name}.png')
    df.to_csv(out / 'submission.csv', index=False)
    # Validate real predictions against the provided converter, not only synthetic patterns.
    import torchvision
    tree = ast.parse((data.root / 'segmentation_to_csv.py').read_text(encoding='utf-8-sig'))
    scope = dict(np=np, pd=pd, Image=Image, torchvision=torchvision, os=os)
    exec(compile(ast.Module(body=[x for x in tree.body if isinstance(x, ast.FunctionDef)], type_ignores=[]),
                 'official_converter', 'exec'), scope)
    reference = scope['turn_to_str']([str(pred_dir / f'{name}.png') for name in df.Id])
    assert reference == df.Predicted.tolist(), '真实预测与官方编码不一致。'
    print('submission.csv：20行，Id顺序、512×512、RLE往返、官方脚本逐字比对全部通过。')
    return df


def bundle_outputs(out, destination):
    """Small analysis bundle; large reproducibility checkpoints remain in Notebook Output."""
    import zipfile
    out=Path(out); destination=Path(destination)
    with zipfile.ZipFile(destination,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in out.rglob('*'):
            if p.is_file() and p.suffix.lower() in {'.json','.csv','.md','.png'}:
                z.write(p,p.relative_to(out))
    print('分析包:',destination,'；模型权重另保存在Output/models。')


In [ ]:
%%writefile /kaggle/working/task4_source/training.py
"""Independent-seed training with explicit successful-update accounting and completed-run checks."""
import copy
import hashlib
import json
import math
import time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import runtime as rt

TRAIN_STEPS = 4000
FORMAL_BATCH = 12

def completed_training(out, signature, steps):
    """Verify actual successful optimizer updates, not a budget multiplied by model count."""
    out = Path(out)
    meta = json.loads((out/'completed.json').read_text())
    history = pd.read_csv(out/'history.csv')
    ck = torch.load(out/'best.pt', map_location='cpu', weights_only=False)
    assert meta['signature'] == signature == ck['signature'], '训练签名不匹配。'
    assert signature == hashlib.sha256(json.dumps(ck['spec'], sort_keys=True).encode()).hexdigest()
    assert meta['optimizer_updates'] == meta['requested_updates'] == steps == ck['spec']['steps'], \
        '实际更新不足；禁止把短训练算作正式预算。'
    assert len(history) >= 2 and history.step.iloc[0] == 0 and history.step.iloc[-1] == steps
    assert history.step.is_monotonic_increasing and not history.step.duplicated().any()
    assert history.attempted_batches.iloc[-1] == meta['attempted_batches']
    assert meta['attempted_batches'] == steps+meta['skipped_updates']
    assert np.isfinite(history[['loss','proxy_dice','lr']]).all().all()
    assert meta['best_step'] == ck['step'] and ck['step'] in history.step.values
    assert ck['recipe'] == meta['recipe'] == ck['spec']['recipe'] and ck['fold'] == meta['fold']
    assert abs(meta['best_proxy_dice']-history.proxy_dice.max()) < 1e-10
    assert abs(meta['best_proxy_dice']-float(history.set_index('step').loc[ck['step'], 'proxy_dice'])) < 1e-10
    return out/'best.pt', meta

def train_seed_fold(data, recipe, fold, out, code_hash,
                    steps=TRAIN_STEPS, batch=FORMAL_BATCH, workers=2, eval_every=250):
    """Scratch-training recipe with verified successful-update accounting."""
    assert steps > 0 and batch > 0 and eval_every > 0
    out = Path(out)
    _, spec = rt.signature(data, recipe, fold, steps, batch, code_hash, eval_every)
    spec.update(initialization='independent_random_seed', warm_start=False,
        objective='original_bce_dice', bn_statistics='training', checkpoint_metric='no_tta_proxy_dice',
        python=rt.platform.python_version(), numpy=np.__version__, opencv=rt.cv2.__version__,
        cuda=torch.version.cuda, cudnn=torch.backends.cudnn.version(),
        gpu=torch.cuda.get_device_name() if rt.DEVICE.type == 'cuda' else None)
    sig = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
    out.mkdir(parents=True, exist_ok=True)
    if (out/'completed.json').exists():
        return completed_training(out, sig, steps)
    if (out/'spec.json').exists():
        assert json.loads((out/'spec.json').read_text())['signature'] == sig, '配置变化，请换OUTPUT_NAME。'
    rt.write_json(out/'spec.json', dict(signature=sig, spec=spec))
    tr, va = np.flatnonzero(data.folds != fold), np.flatnonzero(data.folds == fold)
    assert len(tr) == 16 and len(va) == 4 and not set(tr) & set(va)
    rt.seed_all(recipe.seed)
    model = rt.VesselUNet(recipe).to(rt.DEVICE)
    initial_sha = hashlib.sha256(b''.join(p.detach().cpu().numpy().tobytes()
        for p in model.state_dict().values())).hexdigest()
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    max_skips = 50
    dataset = rt.PatchDataset(data.prepared(recipe.mode), data.labels, tr, recipe,
        (steps+max_skips)*batch, recipe.seed+fold*1000003)
    loader = rt.DataLoader(dataset, batch_size=batch, shuffle=False, num_workers=workers,
        pin_memory=rt.DEVICE.type == 'cuda', persistent_workers=workers > 0)
    optim = torch.optim.AdamW(model.parameters(), lr=recipe.lr, weight_decay=recipe.wd)
    scaler = torch.amp.GradScaler('cuda', enabled=rt.DEVICE.type == 'cuda')
    warmup = min(100, max(1, steps//10))
    start, updates, attempts, skipped = time.time(), 0, 0, 0
    native, best = rt.validate(ema, data, va, recipe.mode)
    best_step = 0
    history = [dict(step=0, attempted_batches=0, loss=0., lr=0., native_dice=native,
        proxy_dice=best, elapsed_seconds=0.)]
    path = out/'best.pt'

    def save_best(chosen, step):
        torch.save(dict(signature=sig, spec=spec, recipe=asdict(recipe), fold=fold, step=step,
            model={k:v.detach().cpu().clone() for k,v in chosen.state_dict().items()}), path.with_suffix('.tmp'))
        path.with_suffix('.tmp').replace(path)

    save_best(ema, 0)
    pd.DataFrame(history).to_csv(out/'history.csv', index=False)
    ema_state = ema.state_dict()
    print(f'Independent RGB seed={recipe.seed}, fold={fold}: {steps} successful updates, batch={batch}', flush=True)
    for x, target in loader:
        attempts += 1
        step = updates+1
        model.train()
        fraction = max(0., (step-warmup)/max(1, steps-warmup))
        lr = recipe.lr*(step/warmup if step <= warmup else .05+.95*.5*(1+math.cos(math.pi*fraction)))
        for group in optim.param_groups:
            group['lr'] = lr
        x, target = x.to(rt.DEVICE), target.to(rt.DEVICE)
        optim.zero_grad(set_to_none=True)
        with rt.autocast():
            loss = rt.loss_fn(model(x), target, recipe.dice_weight)
        assert torch.isfinite(loss), '非有限loss，停止并保留日志。'
        scaler.scale(loss).backward()
        scaler.unscale_(optim)
        norm = nn.utils.clip_grad_norm_(model.parameters(), 5.)
        if rt.DEVICE.type != 'cuda':
            assert torch.isfinite(norm), '非有限梯度。'
        old_scale = scaler.get_scale()
        scaler.step(optim)
        scaler.update()
        if scaler.get_scale() < old_scale:
            skipped += 1
            assert skipped < max_skips, 'AMP跳过过多更新；未写入完成标记。'
            continue
        updates += 1
        decay = min(.995, (updates-1)/(updates+9))
        with torch.no_grad():
            for key, value in model.state_dict().items():
                if value.is_floating_point() and 'running_' not in key:
                    ema_state[key].mul_(decay).add_(value, alpha=1-decay)
                else:
                    ema_state[key].copy_(value)
        if updates % eval_every == 0 or updates == steps:
            native, score = rt.validate(ema if recipe.ema else model, data, va, recipe.mode)
            history.append(dict(step=updates, attempted_batches=attempts, loss=float(loss.detach().cpu()),
                lr=lr, native_dice=native, proxy_dice=score, elapsed_seconds=time.time()-start))
            pd.DataFrame(history).to_csv(out/'history.csv', index=False)
            if score > best:
                best, best_step = score, updates
                save_best(ema if recipe.ema else model, updates)
            print(f'  actual={updates}/{steps} attempted={attempts} loss={history[-1]["loss"]:.4f} proxy={score:.6f}', flush=True)
        if updates == steps:
            break
    assert updates == steps
    meta = dict(signature=sig, recipe=asdict(recipe), fold=int(fold), requested_updates=steps,
        optimizer_updates=updates, attempted_batches=attempts, skipped_updates=skipped,
        best_step=best_step, best_proxy_dice=best, initial_state_sha256=initial_sha,
        warm_start=False, seconds=time.time()-start)
    rt.write_json(out/'completed.json', meta)
    del loader, model, ema, optim
    if rt.DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return completed_training(out, sig, steps)


In [ ]:
%%writefile /kaggle/working/task4_source/predictions.py
"""Fold-aligned probability caches, fixed postprocessing and per-image validation metrics."""
import hashlib
import json
from dataclasses import asdict
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import runtime as rt

CALIBRATION = dict(threshold=.425, min_size=8, output_protocol='native_then_official')

def image_metrics(data, probs, indices, route):
    rows = []
    for i, prob in zip(indices, probs):
        mask = rt.output_mask(prob, **CALIBRATION)
        y = data.proxy[i].astype(bool)
        tp = np.logical_and(y, mask).sum()
        rows.append(dict(route=route, Id=data.ids[i], fold=int(data.folds[i]),
            dice=rt.dice_score(y, mask), native_dice=rt.dice_score(data.labels[i], prob >= .425),
            precision=float(tp/max(1, mask.sum())), recall=float(tp/max(1, y.sum()))))
    return rows

def expanded_predictions(data, recipe, fold, checkpoint, out, code_hash, test=False):
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    indices = list(range(20)) if test else np.flatnonzero(data.folds == fold).tolist()
    ids = data.test_ids if test else [data.ids[i] for i in indices]
    spec = dict(checkpoint_sha256=rt.digest_file(checkpoint), data=data.fingerprint,
                recipe=asdict(recipe), fold=fold, test=test, code_hash=code_hash,
                tta='original_whole_flip4', torch=torch.__version__, numpy=np.__version__,
                opencv=rt.cv2.__version__, device=str(rt.DEVICE))
    signature = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
    path = out/('test.npz' if test else 'oof.npz')
    if path.exists():
        with np.load(path, allow_pickle=False) as cache:
            assert cache['signature'].item() == signature and cache['ids'].tolist() == ids
            probabilities = cache['probs'].copy()
    else:
        model, record = rt.load_model(checkpoint)
        assert record['recipe'] == asdict(recipe) and record['fold'] == fold
        assert record['spec']['data'] == data.fingerprint and record['spec']['code_hash'] == code_hash
        assert record['signature'] == hashlib.sha256(json.dumps(record['spec'], sort_keys=True).encode()).hexdigest()
        images = data.prepared(recipe.mode, test=test)
        probabilities = np.stack([rt.predict(model, images[i], True) for i in indices])
        del model
        torch.cuda.empty_cache()
        with path.with_suffix('.tmp').open('wb') as stream:
            np.savez_compressed(stream, probs=probabilities, ids=ids, signature=signature)
        path.with_suffix('.tmp').replace(path)
    assert probabilities.shape == (len(ids), 584, 565) and probabilities.dtype == np.float32
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    rt.write_json(out/('test.json' if test else 'oof.json'), dict(signature=signature, spec=spec))
    return probabilities


def bundle_expanded_outputs(out, destination):
    import zipfile
    rt.bundle_outputs(out, destination)
    probability_path = Path(out)/'oof_route_probabilities.npz'
    if probability_path.exists():
        with zipfile.ZipFile(destination, 'a', compression=zipfile.ZIP_DEFLATED) as archive:
            archive.write(probability_path, probability_path.name)


In [ ]:
%%writefile /kaggle/working/task4_source/gamma.py
"""Paired channel-wise gamma training augmentation with unchanged raw-RGB inference."""
import hashlib
import json
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import runtime as rt
import predictions as metrics
from training import train_seed_fold
from predictions import expanded_predictions, bundle_expanded_outputs

TRAIN_STEPS = 4000
FORMAL_BATCH = 12
GAMMA_PROBABILITY = .5
GAMMA_RANGE = (.5, 2.)
PILOT_FOLDS = [0, 2, 4]
CONFIRM_FOLDS = [1, 3]
BASE_PATCH_DATASET = rt.PatchDataset


def gamma_chw(image, seed, index):
    """A separate RNG leaves original crop/geometry/photometric draws unchanged."""
    image = np.asarray(image)
    assert image.ndim == 3 and image.shape[0] == 3 and image.dtype == np.float32
    assert np.isfinite(image).all() and image.min() >= -1 and image.max() <= 1
    rng = np.random.default_rng([int(seed), int(index), 20261104])
    if rng.random() >= GAMMA_PROBABILITY:
        return image.copy()
    powers = np.exp(rng.uniform(np.log(GAMMA_RANGE[0]), np.log(GAMMA_RANGE[1]), (3, 1, 1))).astype(np.float32)
    values = np.clip((image+1)*.5, 0, 1)
    return np.ascontiguousarray(2*np.power(values, powers)-1, dtype=np.float32)


class GammaPatchDataset(BASE_PATCH_DATASET):
    def __getitem__(self, index):
        image, target = super().__getitem__(index)
        assert self.cfg.mode == 'rgb' and self.cfg.augmentation
        augmented = gamma_chw(image.numpy(), self.seed, index)
        return torch.from_numpy(augmented), target


def train_gamma_fold(data, recipe, fold, out, code_hash, workers=2):
    assert rt.DEVICE.type == 'cuda' and Path('/kaggle/working').is_dir()
    original = rt.PatchDataset
    assert original is BASE_PATCH_DATASET
    try:
        rt.PatchDataset = GammaPatchDataset
        return train_seed_fold(data, recipe, fold, out, code_hash,
            steps=TRAIN_STEPS, batch=FORMAL_BATCH, workers=workers)
    finally:
        rt.PatchDataset = original


def gamma_recipes():
    rgb = rt.Recipe(name='rgb_seed2', mode='rgb', seed=20261102)
    return dict(reference=rgb, gamma=replace(rgb, name='rgb_seed2_gamma'),
                first=replace(rgb, name='rgb_seed1', seed=20261031),
                green=rt.Recipe(name='green', arch='unet', mode='green', seed=20261031))


def gamma_fusions(branches):
    reference = .375*branches['first']+.375*branches['reference']+.25*branches['green']
    candidate = .375*branches['first']+.375*branches['gamma']+.25*branches['green']
    return reference, candidate


def gamma_dataset_selfcheck(data, recipes, out):
    assert rt.DEVICE.type == 'cuda' and Path('/kaggle/working').is_dir()
    indices = np.flatnonzero(data.folds != 0)
    images = data.prepared('rgb')
    original = BASE_PATCH_DATASET(images, data.labels, indices, recipes['reference'], 24, 20261102)
    augmented = GammaPatchDataset(images, data.labels, indices, recipes['gamma'], 24, 20261102)
    changed = 0
    for index in range(24):
        before, target_before = original[index]
        after, target_after = augmented[index]
        assert torch.equal(target_before, target_after)
        assert before.shape == after.shape == (3, 192, 192)
        expected = gamma_chw(before.numpy(), augmented.seed, index)
        assert np.array_equal(after.numpy(), expected)
        assert torch.isfinite(after).all() and after.min() >= -1 and after.max() <= 1
        changed += int(not torch.equal(before, after))
    assert 0 < changed < 24
    rt.write_json(Path(out)/'augmentation_selfcheck.json', dict(samples=24, identical_targets=24,
        changed_images=changed, optimizer_updates=0,
        scope='Kaggle-only dataset semantics check; no model or accuracy evaluation'))


def gamma_gate(scores, phase):
    table = scores.set_index('fold').sort_index()
    expected = [0] if phase == 'first' else (list(range(5)) if phase == 'confirmation' else PILOT_FOLDS)
    assert table.index.tolist() == sorted(expected) and np.isfinite(table.to_numpy()).all()
    gains = table.gain
    if phase == 'first':
        return dict(accepted=False, pruned=bool(gains.mean() <= -.003), mean_gain=float(gains.mean()),
                    improved_folds=int((gains > 0).sum()))
    assert phase in ['rgb_pilot', 'fusion_pilot', 'confirmation']
    minimum = .001 if phase == 'rgb_pilot' else .0005
    checks = dict(mean_gain_at_least_minimum=bool(gains.mean() >= minimum),
                  enough_improved_folds=bool((gains > 0).sum() >= (3 if phase == 'confirmation' else 2)))
    if phase == 'confirmation':
        fresh = gains.loc[CONFIRM_FOLDS]
        checks.update(confirmation_mean_positive=bool(fresh.mean() > 0),
                      confirmation_has_improvement=bool((fresh > 0).any()))
    return dict(accepted=all(checks.values()), mean_gain=float(gains.mean()),
                improved_folds=int((gains > 0).sum()), minimum_gain=minimum, checks=checks,
                rejection_reasons=[key for key, value in checks.items() if not value])


def run_gamma_augmentation(data, out, code_hash, workers=2):
    assert rt.DEVICE.type == 'cuda' and Path('/kaggle/working').is_dir(), '只允许在Kaggle GPU训练。'
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    recipes = gamma_recipes()
    spec = dict(protocol='paired_rgb_channel_gamma', data=data.fingerprint, code_hash=code_hash,
        recipes={k: asdict(cfg) for k, cfg in recipes.items()}, steps=TRAIN_STEPS, batch=FORMAL_BATCH,
        workers=workers, gamma_probability=GAMMA_PROBABILITY, gamma_range=list(GAMMA_RANGE),
        gamma_distribution='independent log-uniform per RGB channel; post-original patch augmentation',
        labels_changed=False, inference='unchanged original raw-RGB whole flip4',
        reference_weights=dict(first=.375, reference=.375, green=.25),
        candidate_weights=dict(first=.375, gamma=.375, green=.25),
        first_fold_prune_gain_at_most=-.003, rgb_pilot_minimum_gain=.001,
        fusion_minimum_gain=.0005, calibration=metrics.CALIBRATION,
        python=rt.platform.python_version(), torch=torch.__version__, numpy=np.__version__,
        opencv=rt.cv2.__version__, device=str(rt.DEVICE), cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version(), gpu=torch.cuda.get_device_name())
    signature = hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()
    if (out/'experiment_spec.json').exists():
        assert json.loads((out/'experiment_spec.json').read_text())['signature'] == signature, '配置变化，请换OUTPUT_NAME。'
    rt.write_json(out/'experiment_spec.json', dict(signature=signature, spec=spec))
    gamma_dataset_selfcheck(data, recipes, out)
    branches = {key: np.zeros((20, 584, 565), np.float32) for key in recipes}
    models, trained, decisions = {}, {}, {}

    def train_one(key, fold):
        cfg = recipes[key]
        folder = out/'models'/cfg.name/f'fold{fold}'
        if key == 'gamma':
            checkpoint, meta = train_gamma_fold(data, cfg, fold, folder, code_hash, workers=workers)
        else:
            assert rt.PatchDataset is BASE_PATCH_DATASET
            checkpoint, meta = train_seed_fold(data, cfg, fold, folder, code_hash,
                steps=TRAIN_STEPS, batch=FORMAL_BATCH, workers=workers)
        assert meta['requested_updates'] == meta['optimizer_updates'] == TRAIN_STEPS and not meta['warm_start']
        assert Path(checkpoint).resolve().is_relative_to(out.resolve())
        models[key, fold] = checkpoint
        trained[key, fold] = dict(recipe=cfg.name, seed=cfg.seed, fold=fold, signature=meta['signature'],
            checkpoint_sha256=rt.digest_file(checkpoint), actual_updates=meta['optimizer_updates'],
            attempted_batches=meta['attempted_batches'], skipped_updates=meta['skipped_updates'],
            best_step=meta['best_step'], initial_state_sha256=meta['initial_state_sha256'],
            batch=FORMAL_BATCH, gamma_enabled=key == 'gamma', warm_start=False,
            train_ids=[data.ids[i] for i in np.flatnonzero(data.folds != fold)],
            validation_ids=[data.ids[i] for i in np.flatnonzero(data.folds == fold)])
        rt.write_json(out/'training_provenance.json', list(trained.values()))
        if key == 'gamma':
            assert meta['initial_state_sha256'] == trained['reference', fold]['initial_state_sha256']
        branches[key][data.folds == fold] = expanded_predictions(data, cfg, fold, checkpoint,
            out/'probabilities'/cfg.name/f'fold{fold}', code_hash)

    def compare(reference, candidate, folds, prefix, phase):
        indices = np.flatnonzero(np.isin(data.folds, folds))
        frame = pd.DataFrame(metrics.image_metrics(data, reference[indices], indices, 'reference')+
                             metrics.image_metrics(data, candidate[indices], indices, 'gamma_candidate'))
        assert len(frame) == 8*len(folds) and not frame.duplicated(['route', 'Id']).any()
        frame.to_csv(out/(prefix+'_per_image.csv'), index=False)
        pairs = frame.pivot(index=['fold', 'Id'], columns='route', values='dice')
        assert not pairs.isna().any().any() and pairs.groupby(level='fold').size().eq(4).all()
        pairs['gain'] = pairs.gamma_candidate-pairs.reference
        pairs.reset_index().to_csv(out/(prefix+'_paired.csv'), index=False)
        scores = pairs.groupby('fold')[['reference', 'gamma_candidate', 'gain']].mean().reset_index()
        scores.to_csv(out/(prefix+'_fold_scores.csv'), index=False)
        gate = gamma_gate(scores, phase)
        decisions[prefix] = dict(**gate, folds=list(folds), reference_dice=float(scores.reference.mean()),
                                  candidate_dice=float(scores.gamma_candidate.mean()))
        rt.write_json(out/(prefix+'_selection.json'), decisions[prefix])
        with (out/'oof_route_probabilities.tmp').open('wb') as stream:
            np.savez_compressed(stream, reference=reference[indices], candidate=candidate[indices],
                targets=np.stack([data.proxy[i] for i in indices]), ids=[data.ids[i] for i in indices],
                folds=data.folds[indices], stage=prefix, calibration_json=json.dumps(metrics.CALIBRATION), code_hash=code_hash)
        (out/'oof_route_probabilities.tmp').replace(out/'oof_route_probabilities.npz')
        return gate

    def save_selection(stage, accepted=False, submission_available=False):
        latest = next(reversed(decisions.values()))
        selection = dict(protocol=spec['protocol'], stage=stage, signature=signature,
            route='gamma_fusion' if accepted else None, new_route_accepted=accepted,
            submission_available=submission_available,
            comparison_scope='full_three_branch_fusion' if 'fusion_pilot' in decisions else 'single_rgb_only',
            evaluated_oof_images=4*len(latest['folds']), decisions=decisions,
            baseline_oof=latest['reference_dice'], candidate_oof=latest['candidate_dice'],
            completed_models=len(trained), completed_training_updates=sum(r['actual_updates'] for r in trained.values()),
            external_checkpoint_inputs=0, public_score_verified=False, best_known_public=.86150, **metrics.CALIBRATION)
        rt.write_json(out/'selection.json', selection)
        (out/'试验结果.md').write_text('# 逐通道Gamma增强试验\n\n'+json.dumps(selection, ensure_ascii=False, indent=2)+
            '\n\n仅训练时对RGB种子2增加50%概率的独立逐通道Gamma，指数log-uniform[0.5,2]。标签不变、推理不变。'
            '参考是本轮新训练的双RGB种子方案；OOF参与选优，不是独立测试。实际Public仍需提交确认。\n', encoding='utf8')
        return selection

    train_one('reference', 0)
    train_one('gamma', 0)
    first = compare(branches['reference'], branches['gamma'], [0], 'rgb_first', 'first')
    if first['pruned']:
        return save_selection('rgb_first_rejected')
    for fold in [2, 4]:
        train_one('reference', fold)
        train_one('gamma', fold)
    rgb_pilot = compare(branches['reference'], branches['gamma'], PILOT_FOLDS, 'rgb_pilot', 'rgb_pilot')
    if not rgb_pilot['accepted']:
        return save_selection('rgb_pilot_rejected')
    for fold in PILOT_FOLDS:
        for key in ['first', 'green']:
            train_one(key, fold)
    reference, candidate = gamma_fusions(branches)
    fusion_pilot = compare(reference, candidate, PILOT_FOLDS, 'fusion_pilot', 'fusion_pilot')
    if not fusion_pilot['accepted']:
        return save_selection('fusion_pilot_rejected')
    rt.write_json(out/'frozen_pilot.json', dict(signature=signature, candidate_weights=spec['candidate_weights'],
        calibration=metrics.CALIBRATION, pilot=decisions['fusion_pilot']))
    for fold in CONFIRM_FOLDS:
        for key in ['reference', 'gamma', 'first', 'green']:
            train_one(key, fold)
    reference, candidate = gamma_fusions(branches)
    full = compare(reference, candidate, list(range(5)), 'fusion_confirmation', 'confirmation')
    if not full['accepted']:
        return save_selection('confirmation_rejected')
    selection = save_selection('validation_accepted', accepted=True)
    rt.write_json(out/'frozen_selection.json', selection)
    tests = {}
    for key, cfg in recipes.items():
        total = np.zeros((20, 584, 565), np.float32)
        for fold in range(5):
            total += expanded_predictions(data, cfg, fold, models[key, fold],
                out/'probabilities'/cfg.name/f'fold{fold}', code_hash, test=True)/5
        tests[key] = total
    reference_test, candidate_test = gamma_fusions(tests)
    rt.make_submission(data, reference_test, metrics.CALIBRATION, out/'reference')
    rt.make_submission(data, candidate_test, metrics.CALIBRATION, out/'selected')
    np.savez_compressed(out/'selected_probabilities.npz', oof=candidate, test=candidate_test,
                        train_ids=data.ids, test_ids=data.test_ids)
    return save_selection('complete', accepted=True, submission_available=True)


In [ ]:
%%writefile /kaggle/working/task4_source/train.py
"""Train either fixed final three-branch ensemble from official competition data."""
import argparse
import hashlib
import json
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

import runtime as rt
from training import train_seed_fold
from predictions import CALIBRATION, expanded_predictions, image_metrics
from gamma import gamma_recipes, train_gamma_fold


def require_kaggle():
    if rt.DEVICE.type != 'cuda' or not Path('/kaggle/working').is_dir():
        raise RuntimeError('Run training on Kaggle with GPU enabled.')


def source_hash():
    root = Path(__file__).resolve().parent
    return hashlib.sha256(b''.join(p.name.encode() + p.read_bytes()
                                  for p in sorted(root.glob('*.py')))).hexdigest()


def final_recipes(route):
    if route == 'seed_blend':
        return {'rgb1': rt.Recipe(name='input_rgb', mode='rgb'),
                'rgb2': rt.Recipe(name='input_rgb_seed2', mode='rgb', seed=20261102),
                'green': rt.Recipe(name='model_plain_unet', arch='unet', mode='green')}
    if route == 'gamma':
        recipes = gamma_recipes()
        return {'rgb1': recipes['first'], 'rgb2': recipes['gamma'], 'green': recipes['green']}
    raise ValueError(f'Unknown final route: {route}')


def fuse(branches, route):
    if set(branches) != {'rgb1', 'rgb2', 'green'}:
        raise ValueError('Expected RGB1, RGB2 and green probabilities')
    shapes = {np.asarray(p).shape for p in branches.values()}
    if len(shapes) != 1 or any(not np.isfinite(p).all() or ((p < 0) | (p > 1)).any()
                               for p in branches.values()):
        raise ValueError('Invalid probability arrays')
    if route == 'seed_blend':
        return .75*(.5*branches['rgb1'] + .5*branches['rgb2']) + .25*branches['green']
    if route == 'gamma':
        return .375*branches['rgb1'] + .375*branches['rgb2'] + .25*branches['green']
    raise ValueError(f'Unknown final route: {route}')


def run_final(data, out, route, code_hash, workers=2):
    require_kaggle()
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    recipes = final_recipes(route)
    plan = dict(route=route, data=data.fingerprint, code_hash=code_hash,
                recipes={key: asdict(cfg) for key, cfg in recipes.items()},
                steps=4000, batch=12, workers=workers, calibration=CALIBRATION)
    record = out/'plan.json'
    if record.exists() and json.loads(record.read_text()) != plan:
        raise ValueError('Data or configuration changed; use a new output directory')
    rt.write_json(record, plan)
    oof = {key: np.full((20, 584, 565), np.nan, np.float32) for key in recipes}
    models = {}
    for key, cfg in recipes.items():
        for fold in range(5):
            folder = out/'models'/cfg.name/f'fold{fold}'
            if route == 'seed_blend' and key != 'rgb2':
                checkpoint, _ = rt.train_fold(data, cfg, fold, 4000, 12, folder, code_hash, workers)
            elif route == 'gamma' and key == 'rgb2':
                checkpoint, _ = train_gamma_fold(data, cfg, fold, folder, code_hash, workers)
            else:
                checkpoint, _ = train_seed_fold(data, cfg, fold, folder, code_hash, workers=workers)
            models[key, fold] = checkpoint
            oof[key][data.folds == fold] = expanded_predictions(data, cfg, fold, checkpoint,
                out/'probabilities'/cfg.name/f'fold{fold}', code_hash)
    final_oof = fuse(oof, route)
    rows = image_metrics(data, final_oof, range(20), route)
    pd.DataFrame(rows).to_csv(out/'oof_per_image.csv', index=False)
    rt.write_json(out/'metrics.json', dict(route=route, oof_proxy_dice=float(np.mean([r['dice'] for r in rows])),
                  model_count=15, **CALIBRATION))
    # Fixed routes and calibration are saved before any test prediction is used.
    rt.write_json(out/'frozen_route.json', plan)
    tests = {}
    for key, cfg in recipes.items():
        total = np.zeros((20, 584, 565), np.float32)
        for fold in range(5):
            total += expanded_predictions(data, cfg, fold, models[key, fold],
                out/'probabilities'/cfg.name/f'fold{fold}', code_hash, test=True)/5
        tests[key] = total
    final_test = fuse(tests, route)
    for key in recipes:
        directory = out/'components'/key
        directory.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(directory/'probabilities.npz', oof=oof[key], test=tests[key],
                            train_ids=np.asarray(data.ids, dtype=str), test_ids=np.asarray(data.test_ids, dtype=str),
                            folds=data.folds, data_fingerprint=data.fingerprint, route=route, code_hash=code_hash)
    np.savez_compressed(out/'selected_probabilities.npz', oof=final_oof, test=final_test,
                        train_ids=np.asarray(data.ids, dtype=str), test_ids=np.asarray(data.test_ids, dtype=str),
                        folds=data.folds, data_fingerprint=data.fingerprint, route=route, code_hash=code_hash)
    rt.make_submission(data, final_test, CALIBRATION, out/'selected')
    return final_oof, final_test


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data', default=None)
    parser.add_argument('--out', type=Path, default=Path('/kaggle/working/task4'))
    parser.add_argument('--route', choices=['seed_blend', 'gamma'], default='gamma')
    parser.add_argument('--workers', type=int, default=2)
    args = parser.parse_args(argv)
    require_kaggle()
    if args.workers < 0:
        parser.error('--workers must be nonnegative')
    data = rt.OfficialData(rt.find_data(args.data), args.out/'data_checks')
    rt.verify_official_encoder(data.root, args.out/'data_checks')
    run_final(data, args.out/args.route, args.route, source_hash(), args.workers)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /kaggle/working/task4_source/experiments.py
"""Run the original 16-setting screening or the paired channel-wise Gamma experiment on Kaggle."""
import argparse
from pathlib import Path

import runtime as rt
from gamma import run_gamma_augmentation
from predictions import bundle_expanded_outputs
from train import require_kaggle, source_hash


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data', default=None)
    parser.add_argument('--out', type=Path, default=Path('/kaggle/working/task4_experiments'))
    parser.add_argument('--group', choices=['screening', 'gamma'], default='screening')
    parser.add_argument('--workers', type=int, default=2)
    args = parser.parse_args(argv)
    require_kaggle()
    if args.workers < 0:
        parser.error('--workers must be nonnegative')
    args.out.mkdir(parents=True, exist_ok=True)
    data = rt.OfficialData(rt.find_data(args.data), args.out/'data_checks')
    rt.verify_official_encoder(data.root, args.out/'data_checks')
    if args.group == 'screening':
        rt.run_analysis(data, args.out/'screening', source_hash(), workers=args.workers)
    else:
        directory = args.out/'gamma'
        try:
            run_gamma_augmentation(data, directory, source_hash(), args.workers)
        finally:
            if directory.exists():
                bundle_expanded_outputs(directory, args.out/'gamma_analysis.zip')


if __name__ == '__main__':
    main()


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "/kaggle/working/task4_source/train.py", "--data", "/kaggle/input", "--out", "/kaggle/working/task4", "--route", "seed_blend"], check=True)
